In [4]:
library(readxl)
library(dplyr)
library(tidyr)
library(ggplot2)
library(car)
library(vcd)

NameError: name 'library' is not defined

In [ ]:
data <- read_excel("/Users/jahanvi21/Downloads/Project 3/GitHub/Data/Telco_customer_churn.xlsx")

missing_values <- colSums(is.na(data))
print(missing_values)

str(data)

       CustomerID             Count           Country             State 
                0                 0                 0                 0 
             City          Zip Code          Lat Long          Latitude 
                0                 0                 0                 0 
        Longitude            Gender    Senior Citizen           Partner 
                0                 0                 0                 0 
       Dependents     Tenure Months     Phone Service    Multiple Lines 
                0                 0                 0                 0 
 Internet Service   Online Security     Online Backup Device Protection 
                0                 0                 0                 0 
     Tech Support      Streaming TV  Streaming Movies          Contract 
                0                 0                 0                 0 
Paperless Billing    Payment Method   Monthly Charges     Total Charges 
                0                 0                

In [ ]:
data_clean <- data %>%
  select(
    `Tenure Months`,
    `Monthly Charges`,
    Contract,
    `Internet Service`,
    `Tech Support`,
    `Online Security`,
    Gender,
    `Senior Citizen`,
    Partner,
    `Churn Label`
  ) %>%
  rename(Churn = `Churn Label`) %>%
  drop_na() %>%
  mutate(across(where(is.character), as.factor))

colSums(is.na(data_clean))

Tenure Months  Monthly Charges         Contract Internet Service 
               0                0                0                0 
    Tech Support  Online Security           Gender   Senior Citizen 
               0                0                0                0 
         Partner            Churn 
               0                0

In [ ]:
setwd("/Users/jahanvi21/Downloads/Project 3/GitHub/Results")

In [ ]:
numeric_vars <- c("Tenure Months", "Monthly Charges")
categorical_vars <- c("Churn", "Contract", "Internet Service", "Tech Support", "Online Security", "Gender", "Senior Citizen", "Partner")

num_summary <- data.frame(
  Variable = numeric_vars,
  Mean = c(mean(data_clean$`Tenure Months`, na.rm = TRUE), mean(data_clean$`Monthly Charges`, na.rm = TRUE)),
  SD = c(sd(data_clean$`Tenure Months`, na.rm = TRUE), sd(data_clean$`Monthly Charges`, na.rm = TRUE)),
  Median = c(median(data_clean$`Tenure Months`, na.rm = TRUE), median(data_clean$`Monthly Charges`, na.rm = TRUE)),
  Min = c(min(data_clean$`Tenure Months`, na.rm = TRUE), min(data_clean$`Monthly Charges`, na.rm = TRUE)),
  Max = c(max(data_clean$`Tenure Months`, na.rm = TRUE), max(data_clean$`Monthly Charges`, na.rm = TRUE))
)

cat_list <- lapply(categorical_vars, function(var) {
  tbl <- table(data_clean[[var]])
  data.frame(
    Variable = var,
    Category = names(tbl),
    Count = as.numeric(tbl),
    Percentage = round(as.numeric(tbl) / sum(tbl) * 100, 1)
  )
})

cat_summary <- do.call(rbind, cat_list)

write.csv(num_summary, "univariate_numeric_summary.csv", row.names = FALSE)
write.csv(cat_summary, "univariate_categorical_summary.csv", row.names = FALSE)

In [ ]:
numeric_vars <- c("Tenure Months", "Monthly Charges")
categorical_vars <- c("Contract", "Internet Service", "Tech Support", "Online Security", "Gender", "Senior Citizen", "Partner")

num_results <- data.frame(
  Variable = character(),
  Mean_No = numeric(),
  Mean_Yes = numeric(),
  Test_Used = character(),
  P_Value = numeric(),
  stringsAsFactors = FALSE
)

for (var in numeric_vars) {
  group1 <- data_clean[[var]][data_clean$Churn == "No"]
  group2 <- data_clean[[var]][data_clean$Churn == "Yes"]
  
  g1_sample <- if (length(group1) > 5000) sample(group1, 5000) else group1
  g2_sample <- if (length(group2) > 5000) sample(group2, 5000) else group2
  
  shapiro1 <- shapiro.test(g1_sample)$p.value
  shapiro2 <- shapiro.test(g2_sample)$p.value
  
  if (shapiro1 > 0.05 & shapiro2 > 0.05) {
    test_res <- t.test(group1, group2)
    test_name <- "Two-sample t-test"
  } else {
    test_res <- wilcox.test(group1, group2)
    test_name <- "Mann-Whitney U test"
  }
  
  num_results <- rbind(num_results, data.frame(
    Variable = var,
    Mean_No = mean(group1, na.rm = TRUE),
    Mean_Yes = mean(group2, na.rm = TRUE),
    Test_Used = test_name,
    P_Value = test_res$p.value
  ))
}

cat_results <- data.frame(
  Variable = character(),
  Chi_Square_Statistic = numeric(),
  P_Value = numeric(),
  stringsAsFactors = FALSE
)

for (var in categorical_vars) {
  tbl <- table(data_clean$Churn, data_clean[[var]])
  test_res <- chisq.test(tbl)
  
  cat_results <- rbind(cat_results, data.frame(
    Variable = var,
    Chi_Square_Statistic = as.numeric(test_res$statistic),
    P_Value = test_res$p.value
  ))
}

write.csv(num_results, "bivariate_numeric_tests.csv", row.names = FALSE)
write.csv(cat_results, "bivariate_categorical_tests.csv", row.names = FALSE)

In [ ]:
group_no <- data_clean$`Monthly Charges`[data_clean$Churn == "No"]
group_yes <- data_clean$`Monthly Charges`[data_clean$Churn == "Yes"]

set.seed(123)
g1_sample <- if (length(group_no) > 5000) sample(group_no, 5000) else group_no
g2_sample <- if (length(group_yes) > 5000) sample(group_yes, 5000) else group_yes

shapiro_no <- shapiro.test(g1_sample)
shapiro_yes <- shapiro.test(g2_sample)

print("Shapiro-Wilk Test (Churn = No):")
print(shapiro_no)
print("Shapiro-Wilk Test (Churn = Yes):")
print(shapiro_yes)


levene_res <- leveneTest(`Monthly Charges` ~ Churn, data = data_clean)
print("Levene's Test for Homogeneity of Variance:")
print(levene_res)

h1_test <- wilcox.test(`Monthly Charges` ~ Churn, data = data_clean, conf.int = TRUE, exact = FALSE)
print("Mann-Whitney U Test Results:")
print(h1_test)


z_stat <- qnorm(h1_test$p.value / 2)
n_total <- nrow(data_clean)
r_effect <- abs(z_stat) / sqrt(n_total)
print(paste("Effect Size (r):", round(r_effect, 4)))


p1 <- ggplot(data_clean, aes(x = Churn, y = `Monthly Charges`, fill = Churn)) +
  geom_boxplot(alpha = 0.7, outlier.shape = NA) +
  geom_jitter(width = 0.2, alpha = 0.3, size = 0.5) +
  theme_minimal() +
  labs(title = "Distribution of Monthly Charges by Churn Status",
       x = "Churn Status", y = "Monthly Charges") +
  theme(legend.position = "none")

ggsave("h1_monthly_charges_boxplot.png", p1, width = 6, height = 4, dpi = 300)

[1] "Shapiro-Wilk Test (Churn = No):"

	Shapiro-Wilk normality test

data:  g1_sample
W = 0.91304, p-value < 2.2e-16

[1] "Shapiro-Wilk Test (Churn = Yes):"

	Shapiro-Wilk normality test

data:  g2_sample
W = 0.9284, p-value < 2.2e-16

[1] "Levene's Test for Homogeneity of Variance:"
Levene's Test for Homogeneity of Variance (center = median)
        Df F value    Pr(>F)    
group    1  361.84 < 2.2e-16 ***
      7041                      
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
[1] "Mann-Whitney U Test Results:"

	Wilcoxon rank sum test with continuity correction

data:  Monthly Charges by Churn
W = 3667080, p-value < 2.2e-16
alternative hypothesis: true location shift is not equal to 0
95 percent confidence interval:
 -14.44994 -10.54999
sample estimates:
difference in location 
             -12.45001 

[1] "Effect Size (r): 0.1847"


In [ ]:
contingency_table <- table(data_clean$Contract, data_clean$Churn)
contingency_table

chisq_h2 <- chisq.test(contingency_table)
chisq_h2

cramer_v <- assocstats(contingency_table)$cramer
cramer_v

prop_table <- prop.table(contingency_table, margin = 1) * 100
prop_table

p2 <- ggplot(data_clean, aes(x = Contract, fill = Churn)) +
  geom_bar(position = "fill") +
  scale_y_continuous(labels = scales::percent_format()) +
  scale_fill_manual(values = c("No" = "#F8766D", "Yes" = "#00BFC4")) +
  theme_minimal() +
  labs(title = "Contract Type Distribution by Churn Status",
       x = "Contract Type", y = "Percentage") +
  theme(legend.position = "top")

ggsave("h2_contract_churn_barchart.png", p2, width = 6, height = 4, dpi = 300)

                
                   No  Yes
  Month-to-month 2220 1655
  One year       1307  166
  Two year       1647   48


	Pearson's Chi-squared test

data:  contingency_table
X-squared = 1184.6, df = 2, p-value < 2.2e-16


[1] 0.4101157

                
                        No       Yes
  Month-to-month 57.290323 42.709677
  One year       88.730482 11.269518
  Two year       97.168142  2.831858

In [ ]:
anova_model <- aov(`Tenure Months` ~ Contract, data = data_clean)
anova_summary <- summary(anova_model)[[1]]

h3_anova_results <- data.frame(
  Source = trimws(rownames(anova_summary)),
  Df = anova_summary$Df,
  Sum_Sq = anova_summary$`Sum Sq`,
  Mean_Sq = anova_summary$`Mean Sq`,
  F_Value = anova_summary$`F value`,
  P_Value = anova_summary$`Pr(>F)`
)

h3_group_means <- data_clean %>%
  group_by(Contract) %>%
  summarise(
    Mean_Tenure = mean(`Tenure Months`, na.rm = TRUE),
    SD_Tenure = sd(`Tenure Months`, na.rm = TRUE),
    Median_Tenure = median(`Tenure Months`, na.rm = TRUE)
  )

write.csv(h3_anova_results, "h3_anova_results.csv", row.names = FALSE)
write.csv(h3_group_means, "h3_contract_tenure_group_means.csv", row.names = FALSE)

In [ ]:
tukey_res <- TukeyHSD(anova_model)
tukey_df <- as.data.frame(tukey_res$Contract)
tukey_df$Comparison <- rownames(tukey_df)

write.csv(tukey_df, "h3_tukey_posthoc_results.csv", row.names = FALSE)

In [ ]:
data_clean$Churn <- ifelse(data_clean$Churn == "Yes", 1, 0)

model <- glm(
  Churn ~ Contract + `Tenure Months` + `Monthly Charges` + `Internet Service`,
  data = data_clean,
  family = binomial
)

model_summary <- summary(model)$coefficients

h4_logistic_results <- data.frame(
  Term = rownames(model_summary),
  Estimate = model_summary[, 1],
  Std_Error = model_summary[, 2],
  Z_Value = model_summary[, 3],
  P_Value = model_summary[, 4],
  Odds_Ratio = exp(model_summary[, 1])
)

write.csv(h4_logistic_results, "h4_logistic_regression_results.csv", row.names = FALSE)

In [ ]:
table(data_clean$Contract, data_clean$Churn)
table(data_clean$`Internet Service`, data_clean$Churn)

                
                    0    1
  Month-to-month 2220 1655
  One year       1307  166
  Two year       1647   48

             
                 0    1
  DSL         1962  459
  Fiber optic 1799 1297
  No          1413  113

In [ ]:
data_clean$Churn_Binary <- ifelse(data_clean$Churn == "Yes", 1, 0)

model <- glm(
  Churn_Binary ~ Contract + `Internet Service` + `Tech Support` + `Online Security` + `Senior Citizen` + Partner,
  data = data_clean,
  family = binomial(),
  control = glm.control(maxit = 100, epsilon = 1e-08)
)

model_summary <- summary(model)$coefficients
ci <- exp(confint.default(model))[rownames(model_summary), , drop = FALSE]

h4_logistic_results <- data.frame(
  Term = rownames(model_summary),
  Estimate = model_summary[, 1],
  Std_Error = model_summary[, 2],
  Z_Value = model_summary[, 3],
  P_Value = model_summary[, 4],
  Odds_Ratio = exp(model_summary[, 1]),
  OR_CI_Lower = ci[, 1],
  OR_CI_Upper = ci[, 2]
)

write.csv(h4_logistic_results, "h4_logistic_regression_results.csv", row.names = FALSE)

Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
